# NPZ format

> Dependency-free format: a folder with one compressed `.npz` file per episode.

In [ ]:
#| default_exp data.npz

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Iterable

import numpy as np

from stable_marl.data.format import Format, register_format, validate_write_mode

In [ ]:
#| export
META_FILE = 'meta.json'
EPISODE_GLOB = 'episode_*.npz'


def _episode_file(path: Path, index: int) -> Path:
    return path / f'episode_{index:06d}.npz'


def _read_meta(path: Path) -> dict[str, Any] | None:
    meta = path / META_FILE
    return json.loads(meta.read_text()) if meta.is_file() else None


def _write_meta(path: Path, meta: dict[str, Any]):
    (path / META_FILE).write_text(json.dumps(meta, indent=1, default=str) + '\n')

In [ ]:
#| export
class NpzWriter:
    """
    Writes each episode to ``episode_XXXXXX.npz`` (compressed) in a folder, next to a
    ``meta.json`` holding the dataset metadata (env id and kwargs, number of agents...).
    """
    def __init__(self, path: str | Path, mode: str = 'append', metadata: dict[str, Any] | None = None):
        self.path, self.mode = Path(path), validate_write_mode(mode)
        self.metadata = dict(metadata or {})

    def __enter__(self) -> NpzWriter:
        existing = _read_meta(self.path)
        if existing is not None:
            if existing.get('format') != NpzFormat.name:
                raise ValueError(f"{self.path} holds a dataset that is not in the npz format")
            if self.mode == 'error':
                raise FileExistsError(f"A dataset already exists at {self.path}")
            if self.mode == 'overwrite':
                for file in self.path.glob(EPISODE_GLOB):
                    file.unlink()
            elif any(existing.get(k) != v for k, v in json.loads(json.dumps(self.metadata, default=str)).items()):
                raise ValueError(f"Cannot append to {self.path}: its metadata differs from this run's")
        self.path.mkdir(parents=True, exist_ok=True)
        self._next = len(list(self.path.glob(EPISODE_GLOB)))
        _write_meta(self.path, {'format': NpzFormat.name, **self.metadata})
        return self

    def write_episode(self, episode: dict[str, np.ndarray]):
        file = _episode_file(self.path, self._next)
        tmp = file.with_suffix('.tmp')
        with open(tmp, 'wb') as f:
            np.savez_compressed(f, **episode)
        os.replace(tmp, file)   # no half-written episodes if interrupted
        self._next += 1

    def write_episodes(self, episodes: Iterable[dict[str, np.ndarray]]):
        for episode in episodes:
            self.write_episode(episode)

    def __exit__(self, *exc):
        pass

In [ ]:
#| export
class NpzEpisodes:
    "Reads the episodes of an npz dataset folder."
    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.metadata = _read_meta(self.path)
        if self.metadata is None:
            raise FileNotFoundError(f"No {META_FILE} in {self.path}")
        self.files = sorted(self.path.glob(EPISODE_GLOB))

    @property
    def num_episodes(self) -> int:
        return len(self.files)

    def __len__(self) -> int:
        return self.num_episodes

    def episode(self, index: int, keys: Iterable[str] | None = None) -> dict[str, np.ndarray]:
        "The arrays of episode `index` (only `keys`, if given)."
        with np.load(self.files[index], allow_pickle=False) as data:
            return {key: data[key] for key in (data.files if keys is None else keys)}


@register_format
class NpzFormat(Format):
    "A folder with one compressed ``.npz`` file per episode and a ``meta.json``."
    name = 'npz'

    @classmethod
    def detect(cls, path: str | Path) -> bool:
        meta = _read_meta(Path(path))
        return meta is not None and meta.get('format') == cls.name

    @classmethod
    def open_writer(cls, path, mode='append', metadata=None) -> NpzWriter:
        return NpzWriter(path, mode=mode, metadata=metadata)

    @classmethod
    def open_reader(cls, path) -> NpzEpisodes:
        return NpzEpisodes(path)

### Tests

In [ ]:
import tempfile
from fastcore.test import test_fail
from stable_marl.data.format import load_episodes

ep = lambda T: {'state': np.zeros((T + 1, 4)), 'action': np.ones((T, 2), int), 'length': np.array(T)}
with tempfile.TemporaryDirectory() as tmp:
    with NpzWriter(tmp, metadata={'env_id': 'X'}) as w:
        w.write_episodes([ep(3), ep(5)])
    data = load_episodes(tmp)
    assert isinstance(data, NpzEpisodes) and data.num_episodes == 2 and data.metadata['env_id'] == 'X'
    assert data.episode(1)['action'].shape == (5, 2) and list(data.episode(0, keys=['length'])) == ['length']
    with NpzWriter(tmp, metadata={'env_id': 'X'}) as w: w.write_episode(ep(1))    # append
    assert load_episodes(tmp).num_episodes == 3
    def append_other():
        with NpzWriter(tmp, metadata={'env_id': 'Y'}): pass
    test_fail(append_other, contains='metadata differs')
    def error_mode():
        with NpzWriter(tmp, mode='error'): pass
    test_fail(error_mode, contains='already exists')
    with NpzWriter(tmp, mode='overwrite', metadata={'env_id': 'Y'}) as w: w.write_episode(ep(2))
    assert load_episodes(tmp).num_episodes == 1

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()